# Building a Small Language Model (SLM) — and the Road to ChatGPT-and-Beyond

This notebook is a hands-on, heavily-commented walkthrough of **everything this repo does**,
in the order you should learn it. It runs against the *real* code in this project
(`config.py`, `model.py`, `common.py`, `data/`, `checkpoints/`) — nothing here is a toy
reimplementation kept separate from the repo.

**Sections:**
1. Tokenization — turning text into numbers
2. Data preparation — turning numbers into a trainable dataset
3. Model architecture — a GPT-style decoder-only Transformer, built from scratch
4. The training loop — how the model actually learns
5. Generation — sampling text back out of a model
6. Loading the **real** ~10M-parameter model already trained in this repo (`checkpoints/ten_m`)
7. **The road beyond** — what separates a 10M-parameter model from ChatGPT/GPT-4-class
   systems, and a concrete, ordered roadmap for pushing *this* project further

> **Reality check.** Sections 1–5 train a *tiny demo* model for a couple hundred steps —
> fast enough to run in a notebook (seconds), but not enough to write good English. Section 6
> loads the checkpoint this repo already trained for real (thousands of steps — see
> `config.py`'s `ten_m` preset) so you can see what "the same architecture, more training"
> buys you. Section 7 is about the much bigger gap: base completion model -> aligned
> assistant like ChatGPT.
>
> Run this notebook with the working directory set to the `SLM-10M/` project root, using
> the project's virtualenv (`.venv`) as the kernel, so the imports below resolve.


In [ ]:
# --- Setup -------------------------------------------------------------
# We import the project's own modules (config.py, model.py, common.py)
# rather than re-typing them, so this notebook always matches the real code.
# Run the notebook from the SLM-10M/ folder (or edit PROJECT_ROOT below).
import os, sys, math, pickle, time
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F

PROJECT_ROOT = os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

torch.manual_seed(1337)

print("torch:", torch.__version__)
print("MPS (Apple GPU) available:", torch.backends.mps.is_available())
print("CUDA available:", torch.cuda.is_available())


## 1. Tokenization — text is not numbers, tokens are

Neural networks only understand numbers. A **tokenizer** maps text to a sequence of
integers ("tokens") drawn from a fixed **vocabulary**, and back again.

This repo supports two tokenizer styles (see `data/prepare.py`):
- **character-level** — vocabulary = the unique characters seen (tiny, e.g. ~65 for Shakespeare)
- **GPT-2 byte-pair encoding (BPE)** — vocabulary = 50,257 subword pieces (used for `tinystories`
  and `general`)

BPE is a compromise: common words become a single token, rarer/compound words split into a
few pieces, so the vocabulary stays a manageable size while still being able to represent
*any* input text.


In [ ]:
# tiktoken is the actual library used by data/prepare.py and common.py
import tiktoken
enc = tiktoken.get_encoding("gpt2")

sample = "Once upon a time, a small robot learned to dream."
ids = enc.encode_ordinary(sample)
print("text  :", sample)
print("tokens:", ids)
print("count :", len(ids), "tokens for", len(sample), "characters")
print("back  :", enc.decode(ids))

print("\nsub-word splitting in action:")
for t in ids[:6]:
    print(f"  {t:>6}  ->  {enc.decode([t])!r}")


## 2. Data preparation — from tokens to a trainable dataset

`data/prepare.py` streams a dataset (so you only download as many documents as you ask
for), tokenizes every document with the tokenizer above, and writes **three files** per
dataset under `data/<name>/`:

- `train.bin` — a `uint16` stream of token ids, memory-mapped so RAM usage stays tiny even
  for huge datasets
- `val.bin` — a held-out slice, used only to measure whether the model **generalises**
  (rather than memorises) — the honest signal during training
- `meta.pkl` — `{vocab_size, tokenizer name, ...}` so every other script knows how to
  encode/decode consistently with how the data was built

This repo already prepared `tinystories` for you — let's inspect it directly.


In [ ]:
from common import data_dir

dd = data_dir("tinystories")
with open(os.path.join(dd, "meta.pkl"), "rb") as f:
    meta = pickle.load(f)
print("meta:", meta)

train_data = np.memmap(os.path.join(dd, "train.bin"), dtype=np.uint16, mode="r")
val_data   = np.memmap(os.path.join(dd, "val.bin"),   dtype=np.uint16, mode="r")
print(f"train tokens: {len(train_data):,}   val tokens: {len(val_data):,}")

# Round-trip check: decode a chunk of the raw token stream back to text.
chunk = train_data[:60].tolist()
print("\nfirst 60 tokens decoded:\n", enc.decode(chunk))


A training **example** for a language model is simple: *given these `block_size` tokens,
predict the next one at every position.* We build `(x, y)` pairs where `y` is `x` shifted
left by one token — every position in `x` has its "correct next token" sitting at the same
position in `y`.


In [ ]:
def get_batch(data, block_size, batch_size, device):
    """Sample a random batch of (context, next-token-target) pairs. Same logic as
    get_batch() in train.py."""
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i+1:i+1+block_size].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)

from common import pick_device
device = pick_device()
print("using device:", device)

xb, yb = get_batch(train_data, block_size=32, batch_size=2, device=device)
print("x (input ids):", xb[0].tolist())
print("y (targets)  :", yb[0].tolist())
print("\n-> input token at position 0 is", xb[0,0].item(), "-> the model should predict",
      yb[0,0].item(), "(y is x shifted left by one token)")


## 3. Model architecture — a GPT, from scratch

We import the real `GPT` class from `model.py` (read that file side-by-side with this
section — it is the most important file in the repo). Data flow, top to bottom:

```
token ids
  -> token embedding  (a vector for WHAT the token means)
   + positional embedding (a vector for WHERE it sits in the sequence)
  -> N x Transformer block:
       - causal self-attention  (each token gathers information from itself and
                                  EARLIER tokens only — never the future)
       - MLP                    (each token "thinks" individually about what it gathered)
       (both wrapped in a residual connection + LayerNorm, "pre-norm" style)
  -> final LayerNorm
  -> linear head -> logits over the vocabulary -> softmax -> next-token probabilities
```

Key ideas, briefly:
- **Residual connections** (`x = x + sublayer(x)`) let gradients flow through many stacked
  layers without vanishing, and let each block make a small additive refinement rather than
  having to reconstruct the whole representation.
- **LayerNorm** keeps activations at a stable scale so training doesn't blow up or stall.
- **Weight tying**: the input token-embedding matrix and the output projection matrix are
  the *same* tensor — the model uses one shared notion of "what a token looks like" for
  both reading and writing tokens. This alone saves ~8M parameters at this model's scale.


In [ ]:
from model import GPT
from config import get_preset

mc, tc = get_preset("micro")        # a small preset -> fast to instantiate/inspect here
mc.vocab_size = meta["vocab_size"]  # vocab size is dictated by the DATA, not the preset
model = GPT(mc).to(device)

print(f"parameters (excl. positional embedding): {model.num_params()/1e6:.2f}M")

# For small models, the TOKEN EMBEDDING TABLE (vocab_size x n_embd) often dominates the
# parameter budget, because it needs one row per vocabulary entry no matter how "deep"
# the model is. This is exactly the effect the project README calls out for `ten_m`.
emb_params = mc.vocab_size * mc.n_embd
total_params = sum(p.numel() for p in model.parameters())
print(f"token-embedding table (tied w/ output head): {emb_params/1e6:.2f}M "
      f"({emb_params/total_params:.0%} of all parameters)")


### A minimal, from-scratch look at causal self-attention

Before trusting the `CausalSelfAttention` class in `model.py`, let's hand-roll a tiny,
un-trained version on 5 toy tokens just to *see* what the causal mask does to the
attention weights.


In [ ]:
torch.manual_seed(0)
T, d = 5, 4                                     # 5 toy tokens, 4-dim vectors
x = torch.randn(T, d)
Wq, Wk, Wv = torch.randn(d, d), torch.randn(d, d), torch.randn(d, d)
q, k, v = x @ Wq, x @ Wk, x @ Wv

scores = (q @ k.T) / math.sqrt(d)                        # raw attention scores
causal_mask = torch.tril(torch.ones(T, T))               # 1 = "allowed to look", lower-triangular
scores = scores.masked_fill(causal_mask == 0, float("-inf"))  # block the future
weights = torch.softmax(scores, dim=-1)

torch.set_printoptions(precision=2, sci_mode=False)
print("causal attention weights (rows = query token, cols = key token):\n", weights)
print("\nnotice every row is zero for columns > row index -> token i never attends to token > i.")
print("This is what model.py gets for free via F.scaled_dot_product_attention(..., is_causal=True).")


## 4. The training loop — how the model actually learns

Mirrors `train.py`, condensed here so every step is visible in one cell:

- **Loss**: cross-entropy between the model's predicted next-token distribution and the
  actual next token, averaged over every position in the batch. A perfectly random model
  scores `ln(vocab_size)`; anything meaningfully below that means it's learning structure.
- **Backprop**: `loss.backward()` computes the gradient of the loss with respect to
  *every* weight in the network.
- **AdamW**: an adaptive optimizer that uses those gradients (plus running estimates of
  their mean/variance) to nudge each weight. Weight decay is applied only to 2D matrices
  (not biases/LayerNorm scales) — see `GPT.configure_optimizers` in `model.py`.
- **LR schedule**: linear warmup (avoids early instability) then cosine decay down to
  `min_lr` (lets the model settle into a good minimum instead of bouncing around one).
- **Gradient accumulation**: several small forward/backward passes are summed before one
  optimizer step, simulating a larger batch size without the memory cost of one.
- **Gradient clipping**: caps the gradient's overall size so one unlucky batch can't blow
  up the weights.
- **Checkpointing**: `train.py` saves whenever validation loss improves, and resumes from
  the last checkpoint automatically — you can Ctrl-C and re-run safely.

We only run **`DEMO_ITERS`** steps here (seconds), purely to see the mechanics. Real
training (`ten_m` preset) runs for `max_iters=6000` — see `config.py`.


In [ ]:
def lr_at(it, tc):
    """Linear warmup, then cosine decay to min_lr. Identical to train.py's lr_at()."""
    if it < tc.warmup_iters:
        return tc.learning_rate * (it + 1) / tc.warmup_iters
    ratio = min(1.0, (it - tc.warmup_iters) / max(1, tc.max_iters - tc.warmup_iters))
    coeff = 0.5 * (1.0 + math.cos(math.pi * ratio))
    return tc.min_lr + coeff * (tc.learning_rate - tc.min_lr)

@torch.no_grad()
def estimate_loss(model, data_by_split, block_size, batch_size, device, iters=20):
    """Average loss over a few batches -- the honest train/val progress signal."""
    model.eval()
    out = {}
    for split, data in data_by_split.items():
        losses = torch.zeros(iters)
        for k in range(iters):
            X, Y = get_batch(data, block_size, batch_size, device)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

DEMO_ITERS = 200                 # real training uses thousands of iterations (config.py)
tc.max_iters = DEMO_ITERS
tc.eval_interval = 50
opt = model.configure_optimizers(tc.weight_decay, tc.learning_rate, (tc.beta1, tc.beta2))

model.train()
t0 = time.time()
for it in range(DEMO_ITERS):
    for g in opt.param_groups:
        g["lr"] = lr_at(it, tc)                     # anneal the learning rate every step

    opt.zero_grad(set_to_none=True)
    X, Y = get_batch(train_data, mc.block_size, tc.batch_size, device)
    logits, loss = model(X, Y)                      # forward pass: predict next token everywhere
    loss.backward()                                 # backprop: gradient of loss w.r.t. every weight
    torch.nn.utils.clip_grad_norm_(model.parameters(), tc.grad_clip)
    opt.step()                                       # AdamW: nudge every weight using its gradient

    if it % tc.eval_interval == 0 or it == DEMO_ITERS - 1:
        m = estimate_loss(model, {"train": train_data, "val": val_data},
                           mc.block_size, tc.batch_size, device)
        print(f"iter {it:>4}  lr {opt.param_groups[0]['lr']:.2e}  "
              f"train {m['train']:.3f}  val {m['val']:.3f}")

print(f"\ndemo training done in {time.time()-t0:.1f}s")
print("random-guess baseline loss ~= ln(vocab_size) =", round(math.log(mc.vocab_size), 3))
print("(train/val loss dropping below that baseline means it's genuinely learning structure)")


## 5. Generation — sampling text back out

Generation runs the model forward and samples **one** new token, appends it to the
context, and repeats — this is called **autoregressive decoding**. Two knobs shape the
output (see `GPT.generate` in `model.py`):

- **`temperature`** divides the logits before the softmax. Below 1.0 = sharper, more
  confident (and more repetitive); above 1.0 = flatter, more random/creative.
- **`top_k`** zeroes out every token except the `k` most likely ones before sampling, so
  the model can never pick a wildly unlikely token by chance.

A **base** model like this one continues text — it does not "answer" the way an
instruction-tuned assistant does. That's expected at this stage (see `chat.py`'s
docstring) and is exactly the gap Section 7 is about.


In [ ]:
model.eval()   # disable dropout for inference (also required: MPS's fused attention
               # kernel can't run with dropout active, which is why generate.py/chat.py
               # always call .eval() via common.load_model before generating)

prompt = "Once upon a time"
ids = enc.encode_ordinary(prompt)
x = torch.tensor(ids, dtype=torch.long, device=device)[None, :]

out = model.generate(x, max_new_tokens=60, temperature=0.8, top_k=100)
print(f"[demo model, only {DEMO_ITERS} training steps -- expect rough/incoherent English]\n")
print(enc.decode(out[0].tolist()))


## 6. Loading the *real* ~10M model already trained in this repo

`checkpoints/ten_m/ckpt.pt` was trained properly — thousands of iterations over the full
`tinystories` split (see the `ten_m` preset in `config.py`), via:

```bash
python train.py --preset ten_m --data tinystories
```

Let's load it with the exact same helper `chat.py`/`generate.py`/`serve.py` use
(`common.load_model`) and generate from it, to compare against the 200-step demo above.


In [ ]:
from common import load_model, load_meta, get_codec

real_model, ck = load_model("ten_m", device)
print(f"loaded ten_m checkpoint: iter={ck.get('iter')}  best_val={ck.get('best_val'):.3f}  "
      f"trained_on={ck.get('dataset')!r}")
print(f"parameters: {real_model.num_params()/1e6:.2f}M")

encode, decode = get_codec(load_meta(ck["dataset"]))
ids = encode("Once upon a time")
x = torch.tensor(ids, dtype=torch.long, device=device)[None, :]
out = real_model.generate(x, max_new_tokens=120, temperature=0.8, top_k=200)

print("\n" + decode(out[0].tolist()))


**Compare the two generations above.** Same architecture family, same tokenizer, same
dataset — the difference in fluency comes almost entirely from *how much training the
model actually got* (200 steps vs. thousands). Scale (of training, data, and parameters)
is the single biggest lever in language modeling, which is exactly where Section 7 starts.


## 7. The road beyond — from a 10M base model to something ChatGPT-like

A hosted assistant like ChatGPT differs from the base model above along **three
independent axes**. You can improve any of them without touching the others.

| Axis | This repo, today | What closes the gap |
|---|---|---|
| **Scale** | ~10M params, ~tens of millions of tokens (TinyStories subset) | far more parameters, far more (deduplicated, diverse) training tokens, far more compute |
| **Architecture** | vanilla GPT-2-style block (learned pos-emb, LayerNorm, GELU MLP) | RoPE, RMSNorm, SwiGLU, grouped-query attention, MoE, longer context |
| **Alignment** | none — this is a raw base/completion model | supervised fine-tuning (SFT), preference optimization (RLHF/DPO), safety training |

### 7.1 Scale: pretraining is still the biggest lever

Empirical **scaling laws** (Kaplan et al. 2020; Hoffmann et al. 2022, "Chinchilla") found
that loss falls predictably as a power law in parameters *and* training tokens, and that
for a fixed compute budget there is a roughly optimal ratio between the two — undertrained
large models and overtrained tiny models both waste compute.

| Model | Parameters | Training tokens (approx.) |
|---|---|---|
| `ten_m` (this repo) | ~10M | ~tens of millions |
| GPT-2 small (2019) | 124M | ~10B (WebText) |
| GPT-3 (2020) | 175B | ~300B |
| Chinchilla-optimal 70B (2022) | 70B | ~1.4T |
| GPT-4 / current frontier class | undisclosed, widely estimated in the hundreds of billions+ | multiple trillions |

The practical levers, in order of what you can actually try here:
- **More documents**: raise `--max-docs` in `data/prepare.py`, or switch to the `general`
  dataset for broader (not just children's-story) English.
- **More training steps** and/or a **bigger preset** (`small`, ~30M, in `config.py`).
- **Longer context** (`block_size`) so the model can use more prior text per prediction.
- **Better data curation**: deduplication, quality filtering, and mixing multiple sources
  matter more than raw token count once you're past a certain volume.


### 7.2 Architecture modernizations

`model.py` deliberately uses the *original* 2019 GPT-2 recipe for clarity. Every
production-grade model since has swapped in one or more of these — each is a **drop-in
replacement** you can try one at a time, always comparing validation loss before/after
(exactly the workflow the README's "Where to go next" section recommends):

- **RoPE** (rotary position embedding) instead of a learned absolute positional embedding
  table — rotates query/key vectors by an angle proportional to position instead of adding
  a position vector. Generalises better to sequence lengths longer than anything seen in
  training.
- **RMSNorm** instead of LayerNorm — skips mean-centering, just rescales by root-mean-square.
  Slightly cheaper, and used by LLaMA, Mistral, and most current open models.
- **SwiGLU** instead of GELU in the MLP — a gated activation that tends to reach lower loss
  at the same parameter count.
- **Grouped/multi-query attention** — multiple query heads share fewer key/value heads,
  shrinking the KV-cache and speeding up inference at large scale (matters more as context
  length grows).
- **Mixture-of-Experts (MoE)** — many MLP "experts" per layer, but only a few are active per
  token, so total parameters (and hence knowledge capacity) scale up without a proportional
  increase in *compute per token*.
- **FlashAttention** — a fused, memory-efficient attention kernel. You're already getting
  this for free: `model.py`'s `F.scaled_dot_product_attention(..., is_causal=True)` uses a
  fused kernel under the hood on supported hardware.

Two are implemented below as **standalone, runnable snippets** — study them, then try
swapping them into `model.py` in place of `LayerNorm` / `wpe` and re-training, one change
at a time.


In [ ]:
# Modernization #1: RMSNorm instead of LayerNorm (used in LLaMA, Mistral, Gemma, ...).
# No mean-centering -- just rescale by the root-mean-square, then apply a learned scale.
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return norm * self.weight

rn = RMSNorm(8)
test = torch.randn(2, 8) * 5
print("RMSNorm output (mean~0 not guaranteed, but scale is controlled):\n", rn(test))


In [ ]:
# Modernization #2: RoPE (rotary position embedding) instead of a learned wpe table.
# Instead of ADDING a position vector to the token embedding, RoPE ROTATES the query/key
# vectors by an angle proportional to position -- applied inside attention, not at the
# embedding stage.
def rope_angles(seq_len, dim, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(seq_len).float()
    freqs = torch.outer(t, inv_freq)              # (seq_len, dim/2)
    return torch.cos(freqs), torch.sin(freqs)

def apply_rope(x, cos, sin):
    x1, x2 = x[..., ::2], x[..., 1::2]
    return torch.stack([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).flatten(-2)

cos, sin = rope_angles(seq_len=4, dim=8)
demo_q = torch.randn(4, 8)
print("query vectors, rotated by position:\n", apply_rope(demo_q, cos, sin))
print("\n(to use this: drop `wpe` + the '+ pos_emb' line in GPT.forward, and instead")
print(" apply_rope() to q and k inside CausalSelfAttention.forward, before attention.)")


### 7.3 Alignment: the part that actually makes it "ChatGPT"

Architecture and scale improve a **base model** — it still only ever *completes* text.
Turning that into an assistant that follows instructions, holds a conversation, and
refuses harmful requests takes a separate training pipeline, typically three stages:

1. **Pretraining** (what Sections 1–6 did): next-token prediction on raw text ->
   a *base/completion* model with broad knowledge of language, but no notion of
   "being helpful" or "answering a question."

2. **Supervised Fine-Tuning (SFT)**: continue training — *same* cross-entropy loss as
   pretraining — but now on curated `(instruction, response)` pairs, formatted as one
   string (e.g. `"### Instruction:\n{instruction}\n\n### Response:\n{response}"`). The
   loss is usually masked to zero on the instruction tokens, so the model is only
   penalised for getting the *response* wrong. This is what teaches the model the
   assistant "shape" — question in, helpful answer out.

3. **Preference optimization**: collect pairs of responses to the same prompt, with a
   human (or another model, in RLAIF/"Constitutional AI"-style setups) marking which one
   is *better*. Two common ways to use this signal:
   - **RLHF** (the original ChatGPT recipe): train a separate *reward model* to predict
     the preference judgments, then use reinforcement learning (PPO) to optimize the
     assistant against that reward, while a KL penalty keeps it close to the SFT model
     so it doesn't degenerate.
   - **DPO** (Direct Preference Optimization, now the more common choice): skip the
     reward model and RL loop entirely — optimize directly on preference pairs with a
     closed-form loss.

The two loss functions below are runnable and show exactly what each stage optimizes
(illustrative — they're not wired into this repo's checkpoint, since SFT/preference data
does not exist here yet).


In [ ]:
# Stage 2 -- SFT loss: identical cross-entropy to pretraining, but masked to the
# RESPONSE tokens only (response_mask == 1 there, 0 on the instruction/prompt tokens).
def sft_loss(logits, targets, response_mask):
    losses = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1),
                              reduction="none")
    losses = losses * response_mask.view(-1)
    return losses.sum() / response_mask.sum().clamp(min=1)


# Stage 3 -- DPO loss: given a preferred response y_w and a rejected response y_l to the
# SAME prompt, push the policy to prefer y_w over y_l, regularised against how far it
# drifts from the reference (SFT) model so it can't collapse into degenerate outputs.
# logp_* = SUM of log-probabilities the (policy / reference) model assigns to the whole
# winning (w) / losing (l) response -- one scalar per example.
def dpo_loss(logp_w, logp_w_ref, logp_l, logp_l_ref, beta=0.1):
    pi_logratios  = logp_w - logp_l
    ref_logratios = logp_w_ref - logp_l_ref
    return -F.logsigmoid(beta * (pi_logratios - ref_logratios))

print("sft_loss() and dpo_loss() defined.")
print("Together: pretrain (Sections 1-6) -> SFT -> DPO/RLHF is the standard recipe that")
print("turns a base completion model into an assistant like ChatGPT.")


### 7.4 Everything else a hosted assistant adds on top of the model itself

Even a perfectly aligned model is only part of a system like ChatGPT. The rest:

- **Retrieval-augmented generation (RAG)**: search an external document store at query
  time and feed the results into the context, so the model can be accurate about facts
  it never saw during pretraining (and can cite sources).
- **Tool use / function calling**: the model outputs a structured "call this function"
  action instead of prose; the *system* runs it (calculator, code execution, web search,
  this-repo's own `serve.py`-style API) and feeds the result back in.
- **Long context / memory**: summarizing or retrieving from earlier turns so a
  conversation stays coherent well past the model's raw context window.
- **Safety layers**: input/output moderation classifiers, refusal training baked into
  SFT/DPO data, and red-teaming to find and patch failure modes before deployment.
- **Evaluation**: automated benchmarks (MMLU for knowledge, HumanEval for code, etc.) plus
  human-preference evaluations (pairwise comparisons, Elo-style leaderboards) to know
  whether a change actually helped.
- **Serving efficiency**: KV-caching (reuse past attention keys/values instead of
  recomputing them every token), quantization (int8/int4/GGUF) to shrink memory, batching
  concurrent requests, and sometimes distilling a large model down into a smaller,
  cheaper "student" that mimics it.

`serve.py` in this repo is a minimal seed of that last category — a real HTTP API in
front of the model, extendable with the pieces above.


### 7.5 A concrete, ordered roadmap for *this* project

Go one step at a time, and re-run Section 6's generation (or check validation loss) after
each change so you know whether it actually helped:

1. **Train longer / on more data**: raise `--max-docs` in `data/prepare.py`, or run
   `python train.py --preset ten_m --data tinystories` for the full `max_iters` (this
   notebook only ran a 200-step demo).
2. **Try the `small` preset** (~30M params, same architecture) in `config.py` for more
   capacity.
3. **Swap one architecture piece at a time**: `LayerNorm -> RMSNorm`, `GELU -> SwiGLU`,
   learned `wpe -> RoPE` (both prototyped above) — measure val loss after each.
4. **Grow `block_size`** for longer-range coherence, once the above are stable.
5. **Hand-write a small instruction dataset** (even 200–500 pairs) in the SFT format
   above, and fine-tune the trained `ten_m` checkpoint on it using `sft_loss()`.
6. **(Advanced)** Generate or collect preference pairs and try a short DPO pass with
   `dpo_loss()` on top of the SFT model.
7. **(Optional)** Add a minimal retrieval step in `serve.py` — keyword or embedding search
   over a small local document set, injected into the prompt before generation.
8. Compare every result back to this notebook's Section 6 baseline generation.

**Bottom line**: the algorithms in this repo — attention, transformer blocks,
cross-entropy pretraining — are *exactly* the same algorithms behind ChatGPT and GPT-4.
The gap is almost entirely **scale** (parameters x training tokens x compute) plus a
**post-training pipeline** (SFT + preference optimization) this project hasn't run yet —
not a difference in kind. Every piece of that pipeline is sketched, runnable, and ready to
build on above.
